In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub
!pip install catboost



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
df.drop_duplicates(inplace=True)


In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.drop(columns=['Target'])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

df_scaled = pd.DataFrame(scaled_features, columns=features.columns)
df_scaled['Target'] = df['Target'].values


In [ ]:
# Task 5: Write your code here:
target_counts = df['Target'].value_counts(normalize=True)
print("Target distribution:")
print(target_counts)

if target_counts.min() < 0.4:
    print("Target is imbalanced.")
else:
    print("Target is balanced.")


In [ ]:
# Task 1: Write your code here:
X = df_scaled.drop(columns=['Target'])
y = df_scaled['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)



f1_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(verbose=0, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    score = f1_score(y_test, y_pred)
    f1_scores.append(score)
    print(f"Fold {fold+1} F1 Score: {score:.4f}")

print(f"\nAverage F1 Score across folds: {sum(f1_scores)/len(f1_scores):.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier

model_full = CatBoostClassifier(verbose=0, random_state=42)
model_full.fit(X, y)

importances = model_full.get_feature_importance()
feature_names = X.columns

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp["feature"][:10][::-1], feat_imp["importance"][:10][::-1])
plt.xlabel("Importance")
plt.title("Top 10 Feature Importances")
plt.tight_layout()
plt.show()



In [ ]:
# Task 2: Write your code here:
golden_feature = feat_imp.iloc[0]["feature"]
print("Golden Feature:", golden_feature)

In [ ]:
# Task Bonus: Write your code here:
from sklearn.metrics import accuracy_score

golden_feature = feat_imp.iloc[0]["feature"]
print("Golden Feature:", golden_feature)


X_golden = X[[golden_feature]]

k = 5

kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
splits = kf.split(X_golden, y)


golden_accs = []
full_accs = []

for train_idx, test_idx in splits:
    X_train_g, X_test_g = X_golden.iloc[train_idx], X_golden.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    model_golden = CatBoostClassifier(verbose=0, random_state=42)
    model_golden.fit(X_train_g, y_train)
    y_pred_g = model_golden.predict(X_test_g)
    golden_accs.append(accuracy_score(y_test, y_pred_g))

    X_train_full, X_test_full = X.iloc[train_idx], X.iloc[test_idx]
    model_full_cv = CatBoostClassifier(verbose=0, random_state=42)
    model_full_cv.fit(X_train_full, y_train)
    y_pred_full = model_full_cv.predict(X_test_full)
    full_accs.append(accuracy_score(y_test, y_pred_full))

print("\n=== KFold Accuracy Comparison ===")
print(f"Full Model Accuracy (mean):   {np.mean(full_accs):.4f}  (+/- {np.std(full_accs):.4f})")
print(f"Golden Feature Accuracy (mean): {np.mean(golden_accs):.4f}  (+/- {np.std(golden_accs):.4f})")
